# Layerscomparison
=== SEQ MODEL Layers ===

    > [0] Conv2D: 'layer1'
    > [1] Conv2D: 'layer2'

=== NEW_MODEL Layers ===

    > [0] InputLayer: 'input_1'
    > [1] Conv2D: 'layer1'          ← Kopie von seq_model.layers[0]!
    > [2] Conv2D: 'layer2'          ← Kopie von seq_model.layers[1]!
    > [3] Conv2D: 'conv2d_1'        ← NEUE Layer


In [8]:
from keras import layers, models
import numpy as np

# 1. Trainiertes Sequential Model
seq_model = models.Sequential([
    layers.Input(shape=(9, 9, 3)),  #  not as layer
    layers.Conv2D(3, (3, 3), padding='same', name="layer1"),  # layer index= 0
    layers.Conv2D(3, (3, 3), padding='same', name="layer2")  # layer index= 1
])
seq_model.compile(optimizer='adam', loss='mse')
# seq_model.fit(...)  # Weights trainiert!
seq_model.summary()  # Baut Model + inferiert Shapes!

"""
print("Original Layers:")
for i, layer in enumerate(seq_model.layers):
    print(f"  [{i}] {layer.name}: {layer.output_shape}")
"""

# 2. Layers per INDEX extrahieren (trainierte Weights bleiben erhalten!)
trained_layer1 = seq_model.layers[0]  # layer1
trained_layer2 = seq_model.layers[1]  # layer2

# 3. Neues Functional Model mit trainierten Layers
inputs = layers.Input(shape=(9, 9, 3))
x1 = trained_layer1(inputs)    # Nutzt trainierte Weights von layer1!
x2 = trained_layer2(x1)        # Nutzt trainierte Weights von layer2!

outputs = layers.Conv2D(1, (3,3), padding='same')(x2)
new_model = models.Model(inputs, outputs)
new_model.summary()

# 1. Weights extrahieren (Listen)
weights_old1 = seq_model.layers[0].get_weights()   # [kernel, bias] – beide Arrays
weights_new1 = new_model.layers[1].get_weights()   # Gleiches Format

weights_old2 = seq_model.layers[1].get_weights()
weights_new2 = new_model.layers[2].get_weights()

# 2. Jede Komponente einzeln vergleichen
print("\nWeights gleich?")
print(f"layer1 Kernel: {np.allclose(weights_old1[0], weights_new1[0])}")
print(f"layer1 Bias:   {np.allclose(weights_old1[1], weights_new1[1])}")
print(f"layer2 Kernel: {np.allclose(weights_old2[0], weights_new2[0])}")
print(f"layer2 Bias:   {np.allclose(weights_old2[1], weights_new2[1])}")






Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 9, 9, 3)        │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 9, 9, 3)        │            84 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168 (672.00 B)

 Trainable params: 168 (672.00 B)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 9, 9, 3)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer1 (Conv2D)                 │ (None, 9, 9, 3)        │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 9, 9, 3)        │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 9, 9, 1)        │            28 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 196 (784.00 B)

 Trainable params: 196 (784.00 B)

 Non-trainable params: 0 (0.00 B)


Weights gleich?
layer1 Kernel: True
layer1 Bias:   True
layer2 Kernel: True
layer2 Bias:   True
